# Klebsiella Pneumoniae Antimicrobial Resistance in England - Data Dashboard

### What this data shows

This interactive data dashboard shows rolling monthly antimicrobial resistance and testing data for klebsiella pneumoniae bacteraemia in England.

In [ ]:
from ukhsa_api_wrapper import *
import ipywidgets as wdg
import pandas as pd
import matplotlib.pyplot as plt
import json

In [ ]:
#display settings
%matplotlib inline
plt.rcParams['figure.dpi'] = 100

In [ ]:
#Load initial JSON data

#Metric 1: Percent resistant, rolling month, England
with open('initial_data/kpneumoniae_percent_resistant.json', 'rt') as INFILE:
    percent_resistant = json.load(INFILE)

#Metric 2: Number tested, rolling month, UKHSA Regions
regions = ["East Midlands", "East of England", "London", "North East", "North West", "South East", "South West", "West Midlands", "Yorkshire and Humber"]
number_tested = []

for region in regions:
    with open(f'initial_data/kpneumoniae_number_tested_{region}.json', 'rt') as INFILE:
        number_tested.append(json.load(INFILE))



In [7]:
#Wrangle data

#list of strata from data
strata = []

def extract_by_region(dataset_list):
    """Extracts data from list of JSON files to single python Dict format: {date : {region : total_value ...} ...}"""
    data = {}
    for dataset in dataset_list:
        for entry in dataset:
            date = entry['date']
            region = entry['geography']
            stratum = entry['stratum']
            value = entry['metric_value']
            if date not in data:
                data[date] = {}
            if region not in data[date]:
                data[date][region] = value
            else:
                data[date][region] += value
    return data

def extract_by_stratum(dataset):
    """Extracts data from JSON to python Dict format: {date : {stratum : value ...} ...}"""
    data = {}
    for entry in dataset:
        stratum = entry['stratum']
        if stratum not in strata: #populate strata list
            strata.append(stratum)
        date = entry['date']
        value = entry['metric_value']
        if date not in data:
            data[date] = {}
        data[date][stratum] = value
    return data

def create_data_frame(parsed_json, column_list):
    """Converts python Dict to DataFrame with dates as index and strata as columns"""
    #build date-range for index
    dates = list(parsed_json.keys())
    dates.sort()
    startdate = parse_date(dates[0])
    enddate = parse_date(dates[-1])
    index = pd.date_range(startdate, enddate, freq='MS')
    #create DataFrame with date range as index
    df = pd.DataFrame(index=index, columns=column_list)
    #populate DataFrame
    for date, entry in parsed_json.items():
        pd_date = parse_date(date)
        for col in column_list:
            value = entry.get(col, 0.0)
            df.loc[date, col] = value
    df.fillna(0.0, inplace=True)
    return df

#parse data as DataFrames
resistance_df = create_data_frame(extract_by_stratum(percent_resistant), strata)
tested_df = create_data_frame(extract_by_region(number_tested), regions)

In [8]:
#graph 1: percent resistant

#multiple selection list widget for strata
select_strata = wdg.SelectMultiple(
    options = strata,
    value = strata,
    rows = 7,
    description='Select:',
    disabled=False
)

#range slider widget for date-range
month_range = wdg.SelectionRangeSlider(
    options=resistance_df.index.strftime("%Y-%m"),
    index=(0, len(resistance_df.index)-1),
    description='Date-range:',
    disabled=False,
    continuous_update=False, #reduces flickering
    layout={'width':'500px'}
)

#radio button widget for logY True/False
select_logy=wdg.RadioButtons(
    options=['Linear', 'Logarithmic'],
    value='Linear',
    description='Scale:',
    disabled=False
)

#interactive controls alignment
resistance_controls = wdg.HBox([wdg.VBox([month_range, select_logy]), select_strata])

#plot time-series graph of percent resistant from DataFrame
def resistance_graph(cols, date, log):
    ncols=len(cols)
    if log == 'Logarithmic': #select linear/log scale
        log_bool = True
    else:
        log_bool = False
    if ncols>0:
        ax = resistance_df[list(cols)][date[0]:date[1]].plot(kind='line', logy=log_bool) #filter by strata and date-range
        ax.set_ylabel('Percent resistant')
        ax.set_title('Antimicrobial Resistance by Stratum');
        ax.legend(title='Stratum', loc='center left', bbox_to_anchor=(1, 0.5)) #offset legend
        plt.show()
    else: #error handling for when no stratum selected
        print("\nClick to select data for graph")
        print("\n(CTRL-Click to select more than one category)\n")

#interactive graph widget
resistance_graph=wdg.interactive_output(resistance_graph, {'cols': select_strata, 'date': month_range, 'log': select_logy})

#display alignment
display(wdg.VBox([resistance_graph, resistance_controls]))

This time series graph shows the percentage of each klebsiella pneumoniae stratum testing positive for antimicrobial resistance.

The interactive controls enable you to filter by date-range and stratum, and to plot the data as either a linear or logarithmic graph.

In [10]:
#graph 2: number tested

#radio button widget for cumulative/proportional histogram
select_measure=wdg.RadioButtons(
    options=['Cumulative', 'Proportional'],
    value='Cumulative',
    description='Histogram type:',
    disabled=False
)

#dropdown selection list widget for year
select_year = wdg.Dropdown(
    options = tested_df.index.year.unique(),
    value = tested_df.index.year.unique()[-1], #display most recent year
    description='Select Year:',
    disabled=False
)

#controls alignment
tested_controls = wdg.HBox([select_measure, select_year])

#plot graph from DataFrame
def tested_graph(year, measure):
    df = tested_df[tested_df.index.year == year] #filter by year
    if measure == 'Proportional': #plot each column as proportion of total
        totals = df.sum(axis=1)
        df = df.div(totals, axis=0)*100

    df = df[::-1] #most-recent data on top
    ax = df.plot(kind='barh', stacked=True)
    if measure == 'Proportional': #x-axis label dependent on histogram type
            ax.set_xlabel('Percentage of Total Number Tested (all strata)')
    else:
        ax.set_xlabel('Total Number Tested (all strata)')
    ax.set_title('Testing Numbers by UKHSA Region');
    ax.legend(title='UKHSA Region', loc='center left', bbox_to_anchor=(1, 0.5)) #offset legend
    ax.set_yticklabels(df.index.strftime('%Y-%m'))
    plt.show()

#interactive graph widget
tested_graph=wdg.interactive_output(tested_graph, {'year': select_year, 'measure': select_measure})

#display alignment
display(wdg.VBox([tested_graph, tested_controls]))

This histogram shows the number of samples tested for antimicrobial resistance in each region of England.

The interactive controls enable you to view data for each year, and to plot the data as either a cumulative histogram showing the total testing <br>number for each region, or a proportional histogram showing what percentage each region represents of the total testing number for England.

### Where this data was sourced

All data on this dashboard is sourced from the UK Health Security Agency (UKHSA) [public health dashboard](https://ukhsa-dashboard.data.gov.uk/).

### How recent this data is

The initial data displayed on this dashboard covers the period from the beginning of April 2020 to the beginning of December 2024.<br>
Click the button below to refresh this page with updated UKHSA data.


In [13]:
#Refresh data from API

#API structure for query 1
resistance_structure={
            "theme": "infectious_disease",
            "sub_theme": "antimicrobial_resistance",
            "topic": "K-pneumoniae",
            "geography_type": "Nation",
            "geography": "England",
            "metric": "k-pneumoniae_testing_bacteraemiaPercentResistantRollingMonth" 
        }

#API structure for query 2
tested_structure={
            "theme": "infectious_disease",
            "sub_theme": "antimicrobial_resistance",
            "topic": "K-pneumoniae",
            "geography_type": "UKHSA Region",
            "geography": "Yorkshire and Humber",
            "metric": "k-pneumoniae_testing_bacteraemiaNumberTestedRollingMonth"
          }

def access_api(button):
    """Fetches data from the UKHSA API, saves it as DataFrames and refreshes the graphs"""
    #restyle button for loading
    apibutton.icon = 'clock'
    apibutton.button_style = 'info'
    apibutton.disabled = True
    apibutton.description = 'Fetching data ...'
    try:
        #fetch new data from API
        update_resistant = APIwrapper(**resistance_structure).get_all_pages()
        update_tested = []
        for region in regions:
            tested_structure["geography"] = region
            update_tested.append(APIwrapper(**tested_structure).get_all_pages())
    except:
        #restyle button for unsuccessful fetch
        apibutton.icon = 'unlink'
        apibutton.button_style = 'danger'
        apibutton.description = 'Unsuccessful'
    else:
        #wrangle new data to DataFrames
        global resistance_df
        global tested_df
        resistance_df = create_data_frame(extract_by_stratum(update_resistant), strata)
        tested_df = create_data_frame(extract_by_region(update_tested), regions)
        #refresh graphs
        month_range.options = resistance_df.index.strftime("%Y-%m")
        month_range.index = (0, len(month_range.options)-1)
        select_year.options = tested_df.index.year.unique()
        select_year.value = tested_df.index.year.unique()[-1]
        #restyle button for successful fetch
        apibutton.icon = 'check'
        apibutton.button_style = 'success'
        apibutton.description = 'Updated'

#button widget for data refresh 
apibutton=wdg.Button(
    description='Refresh data',
    disabled=False,
    button_style='', # 'success', 'info', 'warning', 'danger' or ''
    tooltip='Click to download current UKHSA data',
    icon='download'
)

#button callback function
apibutton.on_click(access_api)

display(apibutton)

Button(description='Refresh data', icon='download', style=ButtonStyle(), tooltip='Click to download current UK…